# 02 · 替代数据源补齐（非 Choice）

本 notebook 用于在 Choice 额度不足或 EDB 返回 `no data` 时，从公开/低成本数据源补齐当前 ETF 风格轮动项目所需的部分原始数据。

主流程包含已经验证过能读取的数据源：

- 个股日行情与市值：AkShare 东方财富接口，输出兼容 `stock_daily` 字段。
- SHIBOR 1周、SHIBOR 1年：AkShare `macro_china_shibor_all`。
- 国债到期收益率 1年、10年：AkShare `bond_china_yield`。
- 中债新综合指数：AkShare `bond_new_composite_index_cbond`。

DR007 和美元兑人民币中间价保留为可选指标：代码已支持，但不同网络下稳定性可能不同。若你本机能跑通，只需在参数区把它们加入 `VERIFIED_MACRO_INDICATORS`。

## 1. 环境和依赖检查

运行本节确认项目路径、Python 包和本地模块可用。

In [1]:
import sys
from pathlib import Path
import importlib.util
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RAW = PROJECT_ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW =", RAW)

for pkg in ["akshare", "tushare", "baostock", "pyarrow", "pandas"]:
    spec = importlib.util.find_spec(pkg)
    print(f"{pkg:8s}", "OK" if spec else "MISSING")

PROJECT_ROOT = /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3
RAW = /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw
akshare  OK
tushare  MISSING
baostock MISSING
pyarrow  OK
pandas   OK


## 2. 路径、时间范围和运行参数

默认起点为 `2014-01-01`。终点默认读取项目配置 `config/strategy.yaml` 中的 `backtest.end_date`，当前是 `2026-04-17`。如果之后要滚动更新到更晚日期，可直接修改 `END_DATE`。

注意：部分公开源不支持完整 2014 起点，notebook 会在验证区输出实际覆盖范围。

In [14]:
from src.utils.config import load_yaml

strategy_cfg = load_yaml("strategy")

# 默认按项目复现口径。若要滚动更新，可改成 "2026-12-31" 或今天以前的日期。
START_DATE = "2014-01-01"
END_DATE = strategy_cfg["backtest"]["end_date"]

# 主流程：已验证可读取的宏观指标。
# 重复运行不会重复下载：本地 parquet 可读且覆盖 START_DATE~END_DATE 时，会返回 cache hit。
RUN_MACRO_FETCH = True
VERIFIED_MACRO_INDICATORS = [
    "美元兑人民币_中间价",

]

# 可选指标：代码已支持，但在不同网络下可能连接重置或口径需确认。
# 如果你想抓 DR007 和美元中间价，把下一行改成 True。
INCLUDE_OPTIONAL_MACRO = False
OPTIONAL_MACRO_INDICATORS = [
    "美元兑人民币_中间价",
]
if INCLUDE_OPTIONAL_MACRO:
    VERIFIED_MACRO_INDICATORS = VERIFIED_MACRO_INDICATORS + OPTIONAL_MACRO_INDICATORS

# 个股补齐：默认只跑小样本，避免误触发全市场抓取。
# 生产补齐时，把 STOCK_CODES_MANUAL 换成策略所需股票池，或设置 STOCK_UNIVERSE_FILE。
RUN_STOCK_FETCH = True
STOCK_CODES_MANUAL = ["000001.SZ"]
STOCK_UNIVERSE_FILE = None  # 例如 PROJECT_ROOT / "data" / "manual" / "needed_stock_codes.txt"
MAX_STOCK_CODES_PER_RUN = 1
STOCK_SLEEP_SECONDS = 0.5

# 为避免小样本覆盖正式 stock_daily，默认合并到单独文件。
# 当你确认股票池完整后，可以改为 True，写入 data/raw/stock_daily.parquet。
WRITE_CANONICAL_STOCK_DAILY = False

print("fetch window:", START_DATE, "->", END_DATE)
print("macro indicators:", VERIFIED_MACRO_INDICATORS)
print("manual stock codes:", STOCK_CODES_MANUAL)

fetch window: 2014-01-01 -> 2026-04-17
macro indicators: ['美元兑人民币_中间价']
manual stock codes: ['000001.SZ']


## 3. 数据源函数导入

这里复用项目里的可维护模块，不在 notebook 里复制大段爬虫逻辑。输出 schema 与当前项目约定保持一致。

In [15]:
from src.data.cache import parquet_status, parquet_covers
from src.data.alternative_sources import (
    fetch_macro_akshare,
    fetch_stock_daily_akshare,
    combine_stock_parts,
)

CANONICAL_STOCK_COLUMNS = ["date", "code", "close_adj", "total_mv", "float_mv", "turnover", "amount"]
CANONICAL_MACRO_COLUMNS = ["date", "value"]

FIELD_MAPPING = {
    "stock_zh_a_hist": {
        "日期": "date",
        "收盘": "close_adj",
        "成交额": "amount",
        "换手率": "turnover",
    },
    "stock_value_em": {
        "数据日期": "date",
        "总市值": "total_mv",
        "流通市值": "float_mv",
    },
    "macro": {"日期": "date", "指标值": "value"},
}

FIELD_MAPPING

{'stock_zh_a_hist': {'日期': 'date',
  '收盘': 'close_adj',
  '成交额': 'amount',
  '换手率': 'turnover'},
 'stock_value_em': {'数据日期': 'date', '总市值': 'total_mv', '流通市值': 'float_mv'},
 'macro': {'日期': 'date', '指标值': 'value'}}

## 4. 缓存健康检查

当前项目里存在“文件存在但 parquet 不可读”的情况，所以后续跳过下载前必须检查可读性和日期覆盖，而不是只看文件名。

In [16]:
def summarize_cache(paths):
    rows = []
    for path in paths:
        status = parquet_status(Path(path))
        rows.append({
            "path": str(Path(path).relative_to(PROJECT_ROOT)),
            "exists": status.exists,
            "readable": status.readable,
            "rows": status.rows,
            "error": status.error,
        })
    return pd.DataFrame(rows)

raw_main_files = [
    RAW / "trade_calendar.parquet",
    RAW / "stock_universe.parquet",
    RAW / "stock_daily.parquet",
    RAW / "stock_industry.parquet",
    RAW / "index_daily.parquet",
    RAW / "index_constituents.parquet",
    RAW / "etf_info.parquet",
    RAW / "etf_daily.parquet",
]
raw_main_files += [RAW / f"macro_{key}.parquet" for key in VERIFIED_MACRO_INDICATORS]

cache_report = summarize_cache(raw_main_files)
cache_report

,path,exists,readable,rows,error
0,data/raw/trade_calendar.parquet,True,True,2987,None
1,data/raw/stock_universe.parquet,True,True,5420,None
2,data/raw/stock_daily.parquet,True,True,1881421,None
3,data/raw/stock_industry.parquet,True,True,802160,None
4,data/raw/index_daily.parquet,True,True,48814,None
5,data/raw/index_constituents.parquet,False,False,0,None
6,data/raw/etf_info.parquet,True,True,1566,None
7,data/raw/etf_daily.parquet,True,True,701945,None
8,data/raw/macro_美元兑人民币_中间价.parquet,False,False,0,None


## 5. 宏观数据补齐（已验证主流程）

已验证指标和覆盖说明：

| 指标 | 替代源 | 2014-2026 覆盖判断 |
|---|---|---|
| SHIBOR_1周 | AkShare `macro_china_shibor_all` | 源数据主要从 2017 起；不能保证 2014 连续覆盖 |
| SHIBOR_1年 | AkShare `macro_china_shibor_all` | 源数据主要从 2017 起；不能保证 2014 连续覆盖 |
| 国债到期收益率_1年 | AkShare `bond_china_yield` / 中债网 | 你本机已验证 2014-2026 可读 |
| 国债到期收益率_10年 | AkShare `bond_china_yield` / 中债网 | 你本机已验证 2014-2026 可读 |
| 中债新综合指数 | AkShare `bond_new_composite_index_cbond` | 你本机已验证 2014-2026 可读 |

重复运行不会重复爬取已经完整的文件：`fetch_macro_akshare(..., force=False)` 会先检查 `data/raw/macro_<指标名>.parquet` 是否可读且覆盖 `START_DATE` 到 `END_DATE`。如果完整，会返回 `fetched=False, message="cache hit"`。

每个指标失败时只记录错误，不中断整个 notebook。

In [17]:
macro_results = []

if RUN_MACRO_FETCH:
    for key in VERIFIED_MACRO_INDICATORS:
        try:
            result = fetch_macro_akshare(key=key, start=START_DATE, end=END_DATE, force=False)
            macro_results.append({
                "indicator": key,
                "fetched": result.fetched,
                "rows": result.rows,
                "path": str(result.path.relative_to(PROJECT_ROOT)),
                "message": result.message,
                "error": None,
            })
            print(f"[OK] {key}: rows={result.rows}, fetched={result.fetched}, {result.message}")
        except Exception as exc:
            macro_results.append({
                "indicator": key,
                "fetched": False,
                "rows": 0,
                "path": str((RAW / f"macro_{key}.parquet").relative_to(PROJECT_ROOT)),
                "message": "failed",
                "error": f"{type(exc).__name__}: {exc}",
            })
            print(f"[FAILED] {key}: {type(exc).__name__}: {exc}")
else:
    print("RUN_MACRO_FETCH=False，跳过宏观抓取。")

pd.DataFrame(macro_results)

[OK] 美元兑人民币_中间价: rows=791, fetched=True, 


,indicator,fetched,rows,path,message,error
0,美元兑人民币_中间价,True,791,data/raw/macro_美元兑人民币_中间价.parquet,,None


## 6. 个股数据补齐

原则：优先按策略所需股票池补齐，不默认全市场盲拉。

代码来源优先级：

1. `STOCK_CODES_MANUAL` 手动指定少量股票；
2. `STOCK_UNIVERSE_FILE` 指定 txt/csv/parquet 文件；
3. 如果后续 `index_constituents.parquet` 可读，可从指数成分股读取。

输出字段兼容当前项目：`date, code, close_adj, total_mv, float_mv, turnover, amount`。

注意：AkShare 的历史市值接口通常约从 2018 起，2014-2017 的 `total_mv/float_mv` 可能缺失；价格、成交额、换手率覆盖范围取决于东方财富接口。

In [ ]:
def normalize_code(code):
    code = str(code).strip()
    if not code:
        return None
    if "." in code:
        return code.upper()
    prefix = code[:3]
    if prefix in {"600", "601", "603", "605", "688", "689"}:
        return f"{code}.SH"
    if prefix in {"000", "001", "002", "003", "300", "301"}:
        return f"{code}.SZ"
    if prefix in {"430", "831", "832", "833", "834", "835", "836", "837", "838", "839", "870", "871", "872", "873", "920"}:
        return f"{code}.BJ"
    return code

def read_codes_from_file(path):
    if path is None:
        return []
    path = Path(path)
    if not path.exists():
        print(f"[WARN] universe file 不存在: {path}")
        return []
    if path.suffix == ".parquet":
        status = parquet_status(path)
        if not status.readable:
            print(f"[WARN] universe parquet 不可读: {status.error}")
            return []
        df = pd.read_parquet(path)
        col = "code" if "code" in df.columns else df.columns[0]
        return df[col].dropna().astype(str).tolist()
    if path.suffix in {".csv", ".txt"}:
        if path.suffix == ".csv":
            df = pd.read_csv(path)
            col = "code" if "code" in df.columns else df.columns[0]
            return df[col].dropna().astype(str).tolist()
        return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"[WARN] 不支持的 universe 文件类型: {path.suffix}")
    return []

candidate_codes = []
candidate_codes.extend(STOCK_CODES_MANUAL)
candidate_codes.extend(read_codes_from_file(STOCK_UNIVERSE_FILE))

# 如果指数成分股缓存可读，也可以从中提取策略所需股票池。
index_constituents_path = RAW / "index_constituents.parquet"
if parquet_status(index_constituents_path).readable:
    idx_const = pd.read_parquet(index_constituents_path)
    if "code" in idx_const.columns:
        candidate_codes.extend(idx_const["code"].dropna().astype(str).tolist())

candidate_codes = sorted({normalize_code(c) for c in candidate_codes if normalize_code(c)})
run_codes = candidate_codes[:MAX_STOCK_CODES_PER_RUN]

print(f"candidate stock codes: {len(candidate_codes)}")
print(f"this run: {len(run_codes)}", run_codes[:20])

In [ ]:
stock_results = []
stock_part_dir = RAW / "stock_daily_akshare_parts"

if RUN_STOCK_FETCH:
    for code in run_codes:
        try:
            result = fetch_stock_daily_akshare(
                code=code,
                start=START_DATE,
                end=END_DATE,
                part_dir=stock_part_dir,
                force=False,
                sleep=STOCK_SLEEP_SECONDS,
            )
            stock_results.append({
                "code": result.name,
                "fetched": result.fetched,
                "rows": result.rows,
                "path": str(result.path.relative_to(PROJECT_ROOT)),
                "message": result.message,
                "error": None,
            })
            print(f"[OK] {result.name}: rows={result.rows}, fetched={result.fetched}, {result.message}")
        except Exception as exc:
            stock_results.append({
                "code": code,
                "fetched": False,
                "rows": 0,
                "path": None,
                "message": "failed",
                "error": f"{type(exc).__name__}: {exc}",
            })
            print(f"[FAILED] {code}: {type(exc).__name__}: {exc}")
else:
    print("RUN_STOCK_FETCH=False，跳过个股抓取。")

pd.DataFrame(stock_results)

### 6.1 合并个股分片

默认合并到 `data/raw/stock_daily_akshare.parquet`，避免小样本覆盖正式 `stock_daily.parquet`。

当你确认本次股票池就是策略所需完整股票池时，再把 `WRITE_CANONICAL_STOCK_DAILY=True`，写入当前项目读取器默认使用的 `data/raw/stock_daily.parquet`。

In [ ]:
if RUN_STOCK_FETCH and stock_part_dir.exists():
    stock_output = RAW / ("stock_daily.parquet" if WRITE_CANONICAL_STOCK_DAILY else "stock_daily_akshare.parquet")
    combined = combine_stock_parts(stock_part_dir, stock_output)
    print(f"combined rows={combined.rows}, path={combined.path.relative_to(PROJECT_ROOT)}, {combined.message}")
else:
    print("无个股分片需要合并。")

## 7. 数据完整性验证

验证内容：文件存在、parquet 可读、关键字段、日期范围、缺失年份、前几行样例。

In [ ]:
def validate_time_series(path, required_cols, label):
    path = Path(path)
    status = parquet_status(path)
    row = {
        "label": label,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": status.exists,
        "readable": status.readable,
        "rows": status.rows,
        "date_min": None,
        "date_max": None,
        "missing_cols": None,
        "years": None,
        "error": status.error,
    }
    if not status.readable:
        return row, None
    df = pd.read_parquet(path)
    missing_cols = [c for c in required_cols if c not in df.columns]
    row["missing_cols"] = missing_cols
    if "date" in df.columns and not df.empty:
        dates = pd.to_datetime(df["date"])
        row["date_min"] = dates.min()
        row["date_max"] = dates.max()
        row["years"] = sorted(dates.dt.year.dropna().unique().tolist())
    return row, df.head()

validation_rows = []
samples = {}

for key in VERIFIED_MACRO_INDICATORS:
    row, sample = validate_time_series(RAW / f"macro_{key}.parquet", CANONICAL_MACRO_COLUMNS, key)
    validation_rows.append(row)
    samples[key] = sample

stock_output = RAW / ("stock_daily.parquet" if WRITE_CANONICAL_STOCK_DAILY else "stock_daily_akshare.parquet")
row, sample = validate_time_series(stock_output, CANONICAL_STOCK_COLUMNS, "stock_daily_akshare")
validation_rows.append(row)
samples["stock_daily_akshare"] = sample

validation = pd.DataFrame(validation_rows)
validation

In [ ]:
expected_years = set(range(pd.Timestamp(START_DATE).year, pd.Timestamp(END_DATE).year + 1))

coverage_rows = []
for _, row in validation.iterrows():
    years = set(row["years"] or [])
    coverage_rows.append({
        "label": row["label"],
        "date_min": row["date_min"],
        "date_max": row["date_max"],
        "missing_years": sorted(expected_years - years),
        "readable": row["readable"],
        "missing_cols": row["missing_cols"],
    })

pd.DataFrame(coverage_rows)

In [ ]:
for name, sample in samples.items():
    print("\n" + "=" * 80)
    print(name)
    if sample is None:
        print("不可读或不存在")
    else:
        display(sample)

## 8. 可选/待验证数据源（不影响主流程）

以下指标代码已支持，但建议先单独加入运行，确认你本机网络和口径都没问题：

| 指标 | 候选源 | 当前状态 | 处理建议 |
|---|---|---|---|
| DR007 | AkShare `repo_rate_hist` / 中国货币网 | 代码使用 FDR007 近似，严格 DR007 口径需确认；我这里曾遇到连接重置 | 设置 `INCLUDE_OPTIONAL_MACRO=True` 后单独运行宏观 cell；若失败，用中国货币网/Tushare/手动 CSV 兜底 |
| 美元兑人民币_中间价 | AkShare `macro_china_rmb`，备用 `currency_boc_sina` | 代码已做 fallback；我这里曾遇到连接重置 | 设置 `INCLUDE_OPTIONAL_MACRO=True` 后单独运行宏观 cell；若失败，从外汇交易中心/央行导出 CSV |
| 乘用车销量、电影票房、CCFI/CBFI、三峡流量、生产资料价格指数 | AkShare/官网/手动 CSV | 旧 Choice 缓存不可读，替代源尚未逐项验证 | 后续逐项确认公开接口和口径后再纳入主流程 |

如果只想试其中一个可选指标，也可以直接把参数区改成：

```python
INCLUDE_OPTIONAL_MACRO = False
VERIFIED_MACRO_INDICATORS = ["DR007"]
```

或：

```python
VERIFIED_MACRO_INDICATORS = ["美元兑人民币_中间价"]
```

## 9. 后续运行说明

推荐补齐顺序：

1. 先运行 1-5 节补齐已验证宏观指标；
2. 在第 2 节修改 `STOCK_CODES_MANUAL` 或 `STOCK_UNIVERSE_FILE`，用策略所需股票池补齐个股；
3. 每次只提高 `MAX_STOCK_CODES_PER_RUN` 到一个可控批量，避免公开接口限流；
4. 确认股票池完整后，再设置 `WRITE_CANONICAL_STOCK_DAILY=True` 生成项目默认读取的 `stock_daily.parquet`；
5. 每次补齐后运行第 7 节验证覆盖范围和字段。

当前项目兼容格式：

- 宏观：`data/raw/macro_<指标名>.parquet`，字段 `date,value`。
- 个股：`data/raw/stock_daily.parquet`，字段 `date,code,close_adj,total_mv,float_mv,turnover,amount`。

如果旧文件存在但不可读，不要依赖 notebook 旧缓存；本 notebook 会把不可读文件识别为需要重建。